# JC — Holy OG GPU Colab Factory

One-click entry point for `thebigostreets-byte/jc-the-holy-og`.

**Run order:** connect a GPU runtime, run the cells, authorize Google Drive when prompted, and JC stays synchronized with GitHub while persistent data lives in Drive.

The Colab GPU is for Python/AI/asset-processing workloads. The Three.js game still renders in the browser.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
JC_DRIVE = '/content/drive/MyDrive/JC/Holy-OG'
for name in ['assets','checkpoints','outputs','logs']:
    os.makedirs(os.path.join(JC_DRIVE,name), exist_ok=True)
print('JC Drive workspace:', JC_DRIVE)

In [ ]:
# 2. Sync the real JC repository into Drive
import os, subprocess
REPO = 'https://github.com/thebigostreets-byte/jc-the-holy-og.git'
REPO_DIR = os.path.join(JC_DRIVE, 'repo')
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git','clone','--branch','main',REPO,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',REPO_DIR,'checkout','main'], check=True)
    subprocess.run(['git','-C',REPO_DIR,'reset','--hard','origin/main'], check=True)
print('JC commit:', subprocess.check_output(['git','-C',REPO_DIR,'log','-1','--oneline'],text=True).strip())

In [ ]:
# 4. Install JC asset tooling — Colab-safe Node/Python setup
import os, subprocess, sys, shutil

# Python packages stay in Colab's managed runtime.
subprocess.run([sys.executable,'-m','pip','install','-U','pip','setuptools','wheel'], check=True)
subprocess.run([sys.executable,'-m','pip','install','numpy','pillow','trimesh','pygltflib'], check=True)

# Node dependencies are installed in the temporary Colab VM, not Google Drive.
# This avoids Drive filesystem issues and makes npm setup much faster.
if os.path.exists(os.path.join(REPO_DIR,'package.json')):
    NODE_DIR='/content/jc-node'
    if os.path.exists(NODE_DIR):
        shutil.rmtree(NODE_DIR, ignore_errors=True)
    os.makedirs(NODE_DIR, exist_ok=True)
    subprocess.run(['cp','package.json',os.path.join(NODE_DIR,'package.json')],check=True)
    lock=os.path.join(REPO_DIR,'package-lock.json')
    if os.path.exists(lock):
        subprocess.run(['cp',lock,os.path.join(NODE_DIR,'package-lock.json')],check=True)
    npm_cmd=['npm','ci','--no-audit','--no-fund'] if os.path.exists(os.path.join(NODE_DIR,'package-lock.json')) else ['npm','install','--no-audit','--no-fund']
    result=subprocess.run(npm_cmd,cwd=NODE_DIR,text=True,capture_output=True)
    if result.returncode != 0:
        print('npm ci/install failed; retrying with npm install...')
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        subprocess.run(['npm','install','--no-audit','--no-fund'],cwd=NODE_DIR,check=True)
    print('Node tooling ready:', NODE_DIR)

print('JC asset tooling ready in Colab.')


In [ ]:
# 5. Verify CUDA — automatic GPU diagnostics
import torch, subprocess
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA build:', torch.version.cuda)
try:
    smi=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True).strip()
    print('nvidia-smi:', smi)
except Exception as e:
    print('nvidia-smi check failed:', e)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. In Colab select Runtime > Change runtime type > GPU, then Runtime > Restart session and run all cells again.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('JC GPU READY')


## 6. JC browser preview

Optional. This starts the current JC Vite build through Colab's browser proxy. The browser/WebGL renderer remains in the browser; CUDA is reserved for cloud-side workloads.

In [ ]:
## Drive layout

`My Drive/JC/Holy-OG/`

- `repo/` — synchronized JC source
- `assets/` — generated assets
- `checkpoints/` — model/checkpoint files
- `outputs/` — generated outputs
- `logs/` — logs

Python packages are installed into Colab's managed runtime rather than a Drive-mounted virtual environment.
